<a href="https://colab.research.google.com/github/mdzubayereashraf/Ostad-AI-Engineering-Course-Assignments/blob/main/Assignmen_No_19(TeleSupport_AI_Zubayer).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TeleSupport AI**
### **LLM Structured Extraction & Tool-Calling Assistant**

**Objective:**
Build an AI-powered telecom customer support assistant.

**Key Components:**
1. Structured Customer Data Extraction using Pydantic.
2. Cloud LLM Integration.
3. Local LLM Integration using Ollama.
4. LangChain Tools and AI Agent.
5. Interactive Customer Support Assistant.

In [30]:
!pip install -q openai pydantic

### **Q1.1: Pydantic Schema**

In [31]:
from pydantic import BaseModel, Field
from typing import Literal
import json
import time

class CustomerTicket(BaseModel):

    name: str = Field(
        default="Unknown",
        description="Customer's name. Use Unknown if not provided."
    )

    phone: str = Field(
        default="Unknown",
        description="Customer's phone number. Use Unknown if not provided."
    )

    location: str = Field(
        default="Unknown",
        description="Customer's location. Use Unknown if not provided."
    )

    category: Literal[
        "BALANCE",
        "INTERNET",
        "RECHARGE",
        "SIM",
        "COMPLAINT",
        "OTHER"
    ] = Field(description="Main category of the customer's issue.")

    sentiment: Literal[
        "Positive",
        "Negative",
        "Neutral"
    ] = Field(description="Customer's emotional tone.")

    urgency: int = Field(
        ge=1,
        le=5,
        description="Urgency level from 1 to 5."
    )

    summary: str = Field(
        description="A concise summary of the customer's message."
    )


print(json.dumps(CustomerTicket.model_json_schema(), indent=4))

{
    "properties": {
        "name": {
            "default": "Unknown",
            "description": "Customer's name. Use Unknown if not provided.",
            "title": "Name",
            "type": "string"
        },
        "phone": {
            "default": "Unknown",
            "description": "Customer's phone number. Use Unknown if not provided.",
            "title": "Phone",
            "type": "string"
        },
        "location": {
            "default": "Unknown",
            "description": "Customer's location. Use Unknown if not provided.",
            "title": "Location",
            "type": "string"
        },
        "category": {
            "description": "Main category of the customer's issue.",
            "enum": [
                "BALANCE",
                "INTERNET",
                "RECHARGE",
                "SIM",
                "COMPLAINT",
                "OTHER"
            ],
            "title": "Category",
            "type": "string"
        },
     

### **Schema Validation**

In [32]:
ticket = CustomerTicket(
    category="INTERNET",
    sentiment="Negative",
    urgency=4,
    summary="Customer is experiencing slow internet."
)

print(ticket.model_dump_json(indent=4))

{
    "name": "Unknown",
    "phone": "Unknown",
    "location": "Unknown",
    "category": "INTERNET",
    "sentiment": "Negative",
    "urgency": 4,
    "summary": "Customer is experiencing slow internet."
}


### **Q1.2: OpenRouter API Setup**

In [33]:
from google.colab import userdata
from openai import OpenAI

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    timeout=60.0,
    max_retries=2
)

MODEL_NAME = "baidu/ernie-5.5-flash-preview:free"

print("OpenRouter client initialized successfully!")

OpenRouter client initialized successfully!


### **API Connection Test**

In [34]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: API connection successful."
        }
    ],
    max_tokens=50
)

print(response.choices[0].message.content)

API connection successful.


### **Customer Message**

In [35]:
test_messages = [
    "My name is Sayem, number 01712345678, I live in Uttara. No network since morning, this is unacceptable!",

    "Hi, I'm Nusrat from Chattogram. My SIM stopped working after I dropped my phone. 01812345678",

    "How much balance do I have left? My number is 01912345678.",

    "I was charged twice for the same data pack. Fix this now! - Abir, 01912345678, Mirpur",

    "Thanks, the new 4G speed is amazing! I am very happy with the service."
]

print("Total customer messages:", len(test_messages))

Total customer messages: 5


### **Extraction Function**

In [36]:
def extract_customer_ticket(message: str) -> CustomerTicket:

    system_prompt = f"""
    You are a telecom customer support data extraction system.

    Extract the customer's information and return a JSON object.

    Required schema:
    {json.dumps(CustomerTicket.model_json_schema())}

    Rules:
    1. Return exactly the seven required fields.
    2. Missing name, phone, or location must be "Unknown".
    3. Never invent customer information.
    4. Category must match one of the allowed values.
    5. Sentiment must be Positive, Negative, or Neutral.
    6. Urgency must be an integer between 1 and 5.
    7. Network outage and billing disputes are COMPLAINT.
    8. Balance questions are BALANCE.
    9. SIM malfunction is SIM.
    10. Positive feedback about internet speed is INTERNET.
    11. Summary must be one concise sentence.

    Return only valid JSON. Do not include explanations.
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": message}
        ],
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=800
    )

    result = response.choices[0].message.content

    if not result:
        raise ValueError(
            f"Empty response. Finish reason: "
            f"{response.choices[0].finish_reason}"
        )

    return CustomerTicket.model_validate_json(result)

In [37]:
test_ticket = extract_customer_ticket(test_messages[0])

print(test_ticket.model_dump_json(indent=4))

{
    "name": "Sayem",
    "phone": "01712345678",
    "location": "Uttara",
    "category": "COMPLAINT",
    "sentiment": "Negative",
    "urgency": 4,
    "summary": "Customer reports no network since morning and is dissatisfied."
}


### **All Five Messages Extraction**

In [38]:
extracted_tickets = []

for i, message in enumerate(test_messages, start=1):

    print("\n" + "=" * 60)
    print(f"Customer Message {i}")
    print("=" * 60)

    try:
        ticket = extract_customer_ticket(message)
        extracted_tickets.append(ticket)

        print(ticket.model_dump_json(indent=4))

        time.sleep(2)

    except Exception as e:
        print("Extraction failed:", str(e))


print("\nSTRUCTURED EXTRACTION SUMMARY")
print("-" * 40)
print("Total messages:", len(test_messages))
print("Successfully extracted:", len(extracted_tickets))
print("Failed:", len(test_messages) - len(extracted_tickets))


Customer Message 1
{
    "name": "Sayem",
    "phone": "01712345678",
    "location": "Uttara",
    "category": "COMPLAINT",
    "sentiment": "Negative",
    "urgency": 4,
    "summary": "Customer reports no network since morning and is dissatisfied."
}

Customer Message 2
{
    "name": "Nusrat",
    "phone": "01812345678",
    "location": "Chattogram",
    "category": "SIM",
    "sentiment": "Neutral",
    "urgency": 3,
    "summary": "Nusrat from Chattogram reports that her SIM stopped working after she dropped her phone."
}

Customer Message 3
{
    "name": "Unknown",
    "phone": "01912345678",
    "location": "Unknown",
    "category": "BALANCE",
    "sentiment": "Neutral",
    "urgency": 1,
    "summary": "Customer asks how much balance is left on number 01912345678."
}

Customer Message 4
{
    "name": "Abir",
    "phone": "01912345678",
    "location": "Mirpur",
    "category": "COMPLAINT",
    "sentiment": "Negative",
    "urgency": 4,
    "summary": "Customer Abir reports be

### **Q1.3 — Local LLM Integration using Ollama**

### **Install Ollama**

In [39]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [40]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 126 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 0s (3,145 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [41]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


### **Start Ollama Server**

In [42]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(10)

print("Ollama server started!")

Ollama server started!


### **Download Local Model**

In [43]:
!ollama pull gemma4:e2b

### **Connect Python with Ollama**

In [44]:
from openai import OpenAI

local_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

LOCAL_MODEL = "gemma4:e2b"

print("Connected to local Ollama model!")

Connected to local Ollama model!


### **Test Local Model**

In [45]:
response = local_client.chat.completions.create(
    model=LOCAL_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ],
    max_tokens=100
)

print(response.choices[0].message.content)

Hello, I hope you are having a wonderful day!


### **Local Structured Extraction Function**

In [46]:
def extract_local_ticket(message: str) -> CustomerTicket:

    system_prompt = f"""
    You are a telecom customer support extraction system.

    Extract the customer's information.

    Follow this JSON schema:
    {json.dumps(CustomerTicket.model_json_schema())}

    Rules:
    - Missing name, phone, or location = Unknown.
    - Never invent information.
    - Category must be one of the allowed categories.
    - Sentiment must be Positive, Negative, or Neutral.
    - Urgency must be between 1 and 5.
    - Return exactly seven fields.
    - Return JSON only.
    """

    response = local_client.chat.completions.create(
        model=LOCAL_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": message}
        ],
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=800
    )

    result = response.choices[0].message.content

    if not result:
        raise ValueError("Local model returned empty content.")

    return CustomerTicket.model_validate_json(result)

### **Test All Messages Locally**

In [47]:
local_tickets = []
local_times = []

for i, message in enumerate(test_messages, start=1):

    print("\n" + "=" * 50)
    print(f"LOCAL MODEL - MESSAGE {i}")
    print("=" * 50)

    try:
        start = time.time()

        ticket = extract_local_ticket(message)

        elapsed = time.time() - start

        local_tickets.append(ticket)
        local_times.append(elapsed)

        print(ticket.model_dump_json(indent=4))
        print(f"Processing time: {elapsed:.2f} seconds")

    except Exception as e:
        print("Extraction failed:", str(e))


LOCAL MODEL - MESSAGE 1
{
    "name": "Sayem",
    "phone": "01712345678",
    "location": "Uttara",
    "category": "INTERNET",
    "sentiment": "Negative",
    "urgency": 4,
    "summary": "Customer is experiencing no network connectivity since the morning."
}
Processing time: 4.11 seconds

LOCAL MODEL - MESSAGE 2
{
    "name": "Nusrat",
    "phone": "01812345678",
    "location": "Chattogram",
    "category": "SIM",
    "sentiment": "Negative",
    "urgency": 4,
    "summary": "SIM stopped working after dropping the phone."
}
Processing time: 3.43 seconds

LOCAL MODEL - MESSAGE 3
{
    "name": "Unknown",
    "phone": "01912345678",
    "location": "Unknown",
    "category": "BALANCE",
    "sentiment": "Neutral",
    "urgency": 3,
    "summary": "Customer is inquiring about their current account balance."
}
Processing time: 3.45 seconds

LOCAL MODEL - MESSAGE 4
{
    "name": "Abir",
    "phone": "01912345678",
    "location": "Mirpur",
    "category": "BALANCE",
    "sentiment": "Ne

### **Cloud vs Local Comparison**

In [48]:
print("CLOUD vs LOCAL COMPARISON")
print("-" * 45)

print("Cloud model:", MODEL_NAME)
print("Local model:", LOCAL_MODEL)

print("Cloud successful extractions:", len(extracted_tickets), "/ 5")
print("Local successful extractions:", len(local_tickets), "/ 5")

if local_times:
    print("Average local latency:",
          round(sum(local_times) / len(local_times), 2), "seconds")

CLOUD vs LOCAL COMPARISON
---------------------------------------------
Cloud model: baidu/ernie-5.5-flash-preview:free
Local model: gemma4:e2b
Cloud successful extractions: 5 / 5
Local successful extractions: 5 / 5
Average local latency: 3.76 seconds


### **Q2.1 — LangChain Tools**

### **Install LangChain**

In [49]:
!pip install -q langchain langchain-openai

### **Import Required Libraries**

In [50]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

print("LangChain imported successfully!")

LangChain imported successfully!


### **Create Four Tools**

In [51]:
@tool
def get_balance(phone: str) -> str:
    """Check the current mobile account balance using the customer's phone number."""

    return f"The current balance for {phone} is BDT 125.50."

### **Get Data Packages**

In [52]:
@tool
def get_data_packages(validity_days: int) -> str:
    """Find available mobile internet packages based on validity days."""

    return (
        f"Available {validity_days}-day packages:\n"
        "1. 5 GB - BDT 148\n"
        "2. 10 GB - BDT 248\n"
        "3. 20 GB - BDT 398"
    )

### **Create Support Ticket**

In [53]:
@tool
def create_support_ticket(
    phone: str,
    issue: str,
    priority: int
) -> str:
    """Create a customer support ticket with phone number, issue, and priority."""

    return (
        f"Support ticket created successfully!\n"
        f"Phone: {phone}\n"
        f"Issue: {issue}\n"
        f"Priority: {priority}"
    )

### Check Network Status

In [54]:
@tool
def check_network_status(area: str) -> str:
    """Check mobile network availability and outage information for a given area."""

    return (
        f"Network status for {area}: "
        "All services are currently operational."
    )

### **Test the Four Tools**

In [55]:
print(get_balance.invoke({"phone": "01712345678"}))

print(get_data_packages.invoke({"validity_days": 30}))

print(create_support_ticket.invoke({
    "phone": "01812345678",
    "issue": "Internet service unavailable for two days",
    "priority": 4
}))

print(check_network_status.invoke({"area": "Uttara"}))

The current balance for 01712345678 is BDT 125.50.
Available 30-day packages:
1. 5 GB - BDT 148
2. 10 GB - BDT 248
3. 20 GB - BDT 398
Support ticket created successfully!
Phone: 01812345678
Issue: Internet service unavailable for two days
Priority: 4
Network status for Uttara: All services are currently operational.


# **Q2.2 — Build LangChain AI Agent**

### **Initialize LLM**

In [56]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    max_tokens=1000
)

print("LLM initialized successfully!")

LLM initialized successfully!


### **Register All Tools**

In [57]:
tools = [
    get_balance,
    get_data_packages,
    create_support_ticket,
    check_network_status
]

print("Registered Tools:")

for t in tools:
    print("-", t.name)

Registered Tools:
- get_balance
- get_data_packages
- create_support_ticket
- check_network_status


### **Create AI Agent**

In [58]:
SYSTEM_PROMPT = """
You are a polite and professional telecom customer support assistant.

Your responsibilities:

1. Help customers with mobile balance inquiries.
2. Recommend available internet packages.
3. Create support tickets for customer complaints.
4. Check network status in different areas.

Instructions:

- Greet customers politely.
- Understand the customer's request before responding.
- Use the appropriate tools whenever necessary.
- You may use multiple tools for a single request.
- For balance inquiries, use get_balance.
- For package inquiries, use get_data_packages.
- For service complaints, use create_support_ticket.
- For network inquiries, use check_network_status.
- If the customer reports an internet outage, create a support ticket.
- Use priority between 1 and 5.
- If the customer provides a phone number, use it.
- Never invent customer information.
- For general conversation, respond naturally without using tools.
- Always provide a clear and helpful response.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT
)

print("TeleSupport AI Agent created successfully!")

TeleSupport AI Agent created successfully!


# **Q2.3 — Interactive AI Assistant**

### **Create a Helper Function**

In [59]:
def chat_with_agent(user_message: str):

    result = agent.invoke({
        "messages": [
            {"role": "user", "content": user_message}
        ]
    })

    print("\nAI Assistant:")

    final_message = result["messages"][-1]

    if isinstance(final_message.content, str):
        print(final_message.content)
    else:
        print(final_message.content)

    print("\nTools Used:")

    used_tools = []

    for msg in result["messages"]:
        if msg.type == "tool":
            used_tools.append(msg.name)

    if used_tools:
        for tool_name in used_tools:
            print("-", tool_name)
    else:
        print("No tools used.")

    return result

### **Test Case 1: General Conversation**

In [60]:
chat_with_agent("Hey, how are you?")


AI Assistant:
Hi there! I'm doing great, thanks for asking! 😊

I'm here to help you with anything you need regarding your mobile service — whether it's checking your balance, finding the right internet package, reporting an issue, or checking the network status in your area.

What can I help you with today?

Tools Used:
No tools used.


{'messages': [HumanMessage(content='Hey, how are you?', additional_kwargs={}, response_metadata={}, id='5c7eec48-04d5-419f-914e-9e75f1ecfc28'),
  AIMessage(content="Hi there! I'm doing great, thanks for asking! 😊\n\nI'm here to help you with anything you need regarding your mobile service — whether it's checking your balance, finding the right internet package, reporting an issue, or checking the network status in your area.\n\nWhat can I help you with today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 509, 'total_tokens': 594, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 20, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstr

### **Test Case 2: Check Balance**

In [61]:
chat_with_agent(
    "What's the balance for 01712345678?"
)


AI Assistant:
Thank you for reaching out! The current balance for **01712345678** is **BDT 125.50**.

Is there anything else I can help you with today — such as internet packages, network status, or a support request?

Tools Used:
- get_balance


{'messages': [HumanMessage(content="What's the balance for 01712345678?", additional_kwargs={}, response_metadata={}, id='a2e844f8-dd95-4701-a89f-5fb3e575b0db'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 514, 'total_tokens': 568, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 29, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 256, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'baidu/ernie-5.5-flash-preview:free', 'system_fingerprint': None, 'id': 'gen-1795619804-h578DiK8lgdyw8gmSiYu', 'finish_reason': 'tool_calls', 'logprobs': None, 'normalized_finish_reason': 'tool_call

### **Test Case 3: Multiple Tool Calling**

In [62]:
chat_with_agent(
    "Show me 30-day packages, and my internet has been down "
    "for 2 days, my number is 01812345678."
)


AI Assistant:
Here's what I found for you:

**30-Day Internet Packages:**
1. **5 GB** – BDT 148
2. **10 GB** – BDT 248
3. **20 GB** – BDT 398

**Support Ticket Created:**
- **Phone:** 01812345678
- **Issue:** Internet outage – service down for 2 days
- **Priority:** 4 (High)

Our technical team will look into your outage as soon as possible. Is there anything else I can help you with?

Tools Used:
- get_data_packages
- create_support_ticket


{'messages': [HumanMessage(content='Show me 30-day packages, and my internet has been down for 2 days, my number is 01812345678.', additional_kwargs={}, response_metadata={}, id='413d62ed-8258-4082-8d48-2c301311895c'),
  AIMessage(content="I'm sorry to hear about your internet issue! Let me look up the 30-day packages for you and create a support ticket for your outage right away.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 206, 'prompt_tokens': 530, 'total_tokens': 736, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 100, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 256, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_nam

### **Test Case 4: Network Status**

In [63]:
chat_with_agent(
    "Is there any network problem in Uttara?"
)


AI Assistant:
Good news! I checked the network status for Uttara, and all services are currently operational — there are no reported outages or network problems in your area.

Is there anything else I can help you with today, such as checking your balance, exploring internet packages, or anything else?

Tools Used:
- check_network_status


{'messages': [HumanMessage(content='Is there any network problem in Uttara?', additional_kwargs={}, response_metadata={}, id='60029c24-25aa-431b-8339-ee5109d81388'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 512, 'total_tokens': 562, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 25, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 256, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'baidu/ernie-5.5-flash-preview:free', 'system_fingerprint': None, 'id': 'gen-1795619881-Xi7UWFhCQAB6F9J6o71u', 'finish_reason': 'tool_calls', 'logprobs': None, 'normalized_finish_reason': 'tool_

### **Interactive Chat Loop**

In [70]:
print("=" * 55)
print("WELCOME TO TELESUPPORT AI")
print("Type 'exit' to end the conversation.")
print("=" * 55)

while True:

    user_input = input("\nYou: ").strip()

    if user_input.lower() == "exit":
        print("\nAI: Thank you for contacting TeleSupport AI. Goodbye!")
        break

    if not user_input:
        print("Please enter a message.")
        continue

    try:
        result = agent.invoke({
            "messages": [
                {"role": "user", "content": user_input}
            ]
        })

        print("\nAI:")
        print(result["messages"][-1].content)

        print("\nTools Used:")

        used_tools = [
            msg.name
            for msg in result["messages"]
            if msg.type == "tool"
        ]

        if used_tools:
            for tool_name in used_tools:
                print("-", tool_name)
        else:
            print("No tools used.")

    except Exception as e:
        print("Error:", str(e))

WELCOME TO TELESUPPORT AI
Type 'exit' to end the conversation.

You: hi
Error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day: https://openrouter.ai/settings/credits?utm_source=api&utm_campaign=free_daily_limit', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1795651200000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None}}, 'user_id': 'user_3MawYgEW7zr4YdPFrcImewZB6Yb'}

You: exit

AI: Thank you for contacting TeleSupport AI. Goodbye!


## **Conclusion**

The TeleSupport AI project successfully demonstrates the integration of Large Language Models into a telecom customer support system.

The project includes:

1. Structured customer data extraction using Pydantic.
2. Cloud-based LLM integration for extracting customer information.
3. Local LLM integration using Ollama.
4. Performance comparison between cloud and local models.
5. Four custom LangChain tools for telecom support operations.
6. An AI agent capable of selecting and executing appropriate tools.
7. An interactive customer support assistant.

The system was tested using multiple customer scenarios, including general inquiries, balance checks, internet package recommendations, service complaints, and network status checks.

This project demonstrates how LLMs, structured outputs, and tool calling can be combined to build an intelligent customer support assistant.